In [1]:
!pip install ultralytics opencv-python pandas pyarrow scipy tqdm

In [ ]:
0import cv2
import numpy as np
import pandas as pd
from ultralytics import YOLO
from scipy.optimize import linear_sum_assignment
from tqdm import tqdm
from pathlib import Path

# COCO KEYPOINT INDEXES

L_SHOULDER, R_SHOULDER = 5, 6
L_ELBOW, R_ELBOW = 7, 8
L_WRIST, R_WRIST = 9, 10
L_HIP, R_HIP = 11, 12
L_KNEE, R_KNEE = 13, 14

# HSV COLORS

LOWER_RED1 = np.array([0, 100, 70])
UPPER_RED1 = np.array([12, 255, 255])

LOWER_RED2 = np.array([165, 100, 70])
UPPER_RED2 = np.array([180, 255, 255])

LOWER_BLUE = np.array([90, 100, 70])
UPPER_BLUE = np.array([130, 255, 255])

LOWER_WHITE = np.array([0, 0, 150])
UPPER_WHITE = np.array([180, 60, 255])

# COLOR FEATURES

def extract_colors(crop, mask):

    if crop is None or mask is None:
        return 0.0, 0.0, 0.0

    if crop.shape[:2] != mask.shape[:2]:
        return 0.0, 0.0, 0.0

    if crop.shape[0] == 0 or crop.shape[1] == 0:
        return 0.0, 0.0, 0.0

    try:
        hsv = cv2.cvtColor(crop, cv2.COLOR_BGR2HSV)

        roi_pixels = cv2.bitwise_and(
            hsv,
            hsv,
            mask=mask.astype(np.uint8)
        )

        total_pixels = np.count_nonzero(mask)

        if total_pixels == 0:
            return 0.0, 0.0, 0.0

        mask_red = cv2.bitwise_or(
            cv2.inRange(roi_pixels, LOWER_RED1, UPPER_RED1),
            cv2.inRange(roi_pixels, LOWER_RED2, UPPER_RED2)
        )

        mask_blue = cv2.inRange(
            roi_pixels,
            LOWER_BLUE,
            UPPER_BLUE
        )

        mask_white = cv2.inRange(
            roi_pixels,
            LOWER_WHITE,
            UPPER_WHITE
        )

        return (
            np.sum(mask_red > 0) / total_pixels,
            np.sum(mask_blue > 0) / total_pixels,
            np.sum(mask_white > 0) / total_pixels
        )

    except:
        return 0.0, 0.0, 0.0

def get_polygon_crop(frame, pts):

    if any(p[2] < 0.4 for p in pts):
        return None, None

    pts_np = np.array([[p[0], p[1]] for p in pts], dtype=np.int32)

    x, y, w, h = cv2.boundingRect(pts_np)

    crop = frame[y:y+h, x:x+w]

    if crop.size == 0:
        return None, None

    mask = np.zeros((h, w), dtype=np.uint8)

    shifted = pts_np - [x, y]

    cv2.fillPoly(mask, [shifted], 255)

    return crop, mask

def classify_person(frame, kps):

    total_red = 0
    total_blue = 0

    torso_pts = [
        kps[L_SHOULDER],
        kps[R_SHOULDER],
        kps[R_HIP],
        kps[L_HIP]
    ]

    crop, mask = get_polygon_crop(frame, torso_pts)

    if crop is not None:

        r, b, _ = extract_colors(crop, mask)

        total_red += r
        total_blue += b

    return total_red, total_blue

# ROLE TRACKER

class BoxingRoleManager:

    def __init__(self):

        self.prev_centers = {
            'RED': None,
            'BLUE': None
        }

    def update(self, frame, track_ids, bboxes, keypoints):

        candidates = []

        frame_h, frame_w = frame.shape[:2]
        diag = np.sqrt(frame_h**2 + frame_w**2)

        for tid, bbox, kps in zip(track_ids, bboxes, keypoints):

            red, blue = classify_person(frame, kps)

            cx = (bbox[0] + bbox[2]) / 2
            cy = (bbox[1] + bbox[3]) / 2

            candidates.append({
                'tid': tid,
                'bbox': bbox,
                'kps': kps,
                'center': (cx, cy),
                'red': red,
                'blue': blue
            })

        if len(candidates) == 0:
            return {}

        cost = np.zeros((len(candidates), 2))

        for i, c in enumerate(candidates):

            s_red = c['red']
            s_blue = c['blue']

            if self.prev_centers['RED'] is not None:

                dist = np.linalg.norm(
                    np.array(c['center']) -
                    np.array(self.prev_centers['RED'])
                )

                mem = max(0, 1 - dist / (diag * 0.3))

                s_red = s_red * 0.6 + mem * 0.4

            if self.prev_centers['BLUE'] is not None:

                dist = np.linalg.norm(
                    np.array(c['center']) -
                    np.array(self.prev_centers['BLUE'])
                )

                mem = max(0, 1 - dist / (diag * 0.3))

                s_blue = s_blue * 0.6 + mem * 0.4

            cost[i, 0] = 1 - s_red
            cost[i, 1] = 1 - s_blue

        rows, cols = linear_sum_assignment(cost)

        fighters = {}

        for r, c in zip(rows, cols):

            role = 'RED' if c == 0 else 'BLUE'

            fighters[role] = candidates[r]

            self.prev_centers[role] = candidates[r]['center']

        return fighters

# VIDEO PROCESSING

model = YOLO("yolo11m-pose.pt")

def process_video(video_path, output_path):

    cap = cv2.VideoCapture(str(video_path))

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    manager = BoxingRoleManager()

    rows = []

    pbar = tqdm(total=total_frames)

    frame_idx = 0

    while cap.isOpened():

        ret, frame = cap.read()

        if not ret:
            break

        results = model.track(
            frame,
            persist=True,
            classes=[0],
            conf=0.35,
            verbose=False
        )

        if (
            results[0].boxes is not None
            and results[0].boxes.id is not None
        ):

            bboxes = results[0].boxes.xyxy.cpu().numpy()
            track_ids = results[0].boxes.id.int().cpu().numpy()
            keypoints = results[0].keypoints.data.cpu().numpy()

            fighters = manager.update(
                frame,
                track_ids,
                bboxes,
                keypoints
            )

            for role in ['RED', 'BLUE']:

                if role not in fighters:
                    continue

                fighter = fighters[role]

                row = {
                    'frame': frame_idx,
                    'fighter': role.lower(),
                    'track_id': int(fighter['tid'])
                }

                bbox = fighter['bbox']

                row['x1'] = float(bbox[0])
                row['y1'] = float(bbox[1])
                row['x2'] = float(bbox[2])
                row['y2'] = float(bbox[3])

                for i, kp in enumerate(fighter['kps']):

                    row[f'kp_{i}_x'] = float(kp[0])
                    row[f'kp_{i}_y'] = float(kp[1])
                    row[f'kp_{i}_c'] = float(kp[2])

                rows.append(row)

        frame_idx += 1
        pbar.update(1)

    pbar.close()
    cap.release()

    df = pd.DataFrame(rows)

    df.to_parquet(output_path)


MODE = "tournament_1_part1"

# MAIN

VIDEOS_CSV = "test/videos.csv"

VIDEOS_ROOT = Path("videos_test")

OUTPUT_DIR = Path("outputs_test/poses")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

videos_df = pd.read_csv(VIDEOS_CSV)

# SPLITS


print(f"Videos to process: {len(videos_df)}")

# PROCESS

for _, row in videos_df.iterrows():

    video_key = row["video_key"]

    relative_path = row["video_path"]

    full_video_path = VIDEOS_ROOT / relative_path

    output_path = OUTPUT_DIR / f"{video_key}.parquet"

    print("\n======================")
    print(video_key)

    if output_path.exists():

        print("Already processed")
        continue

    try:

        process_video(
            full_video_path,
            output_path
        )

        print("DONE")

    except Exception as e:

        print("ERROR:", e)

Videos to process: 9

agn_037


100%|██████████| 7952/7952 [11:13<00:00, 11.81it/s]


DONE

agn_038


100%|██████████| 7983/7983 [12:20<00:00, 10.78it/s]


DONE

agn_039


...

DONE

agn_063


100%|██████████| 7044/7044 [10:27<00:00, 11.23it/s]  


DONE

agn_064


100%|██████████| 6831/6831 [10:32<00:00, 10.81it/s]


DONE
